In [1]:
from dotenv import load_dotenv
from langchain_ollama.embeddings import OllamaEmbeddings
import os

load_dotenv()

OLLAMA_BASEURL = os.getenv("OLLAMA_BASEURL")
OLLAMA_MODEL_NAME = os.getenv("OLLAMA_MODEL_NAME")
OLLAMA_EMBEDDING_MODEL_NAME = (
    "bge-m3:latest"  # os.getenv("OLLAMA_EMBEDDING_MODEL_NAME")
)

embeddings = OllamaEmbeddings(
    model=OLLAMA_EMBEDDING_MODEL_NAME, base_url=OLLAMA_BASEURL
)

/home/cooper/githubProjects/agent-deploy-kit/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_core.documents import Document

docs = [
    Document(page_content="FAISS 是向量检索库", metadata={"source": "wiki/faiss"}),
    Document(page_content="LangChain 是 LLM 编排框架", metadata={"source": "wiki/lc"}),
]

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)

In [12]:
vector_store.save_local("faiss_index")

In [13]:
vector_store = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True,  # 反序列化 pickle 需要显式确认
)

In [14]:
results = vector_store.similarity_search("什么是向量检索", k=3)
results

[Document(id='5b208e74-df21-4596-940a-a033f830f77e', metadata={'source': 'wiki/faiss'}, page_content='FAISS 是向量检索库'),
 Document(id='884c22ba-1443-4835-b363-11bbb41b6811', metadata={'source': 'wiki/lc'}, page_content='LangChain 是 LLM 编排框架')]

In [15]:
results = vector_store.similarity_search_with_score("什么是向量检索", k=3)
results

[(Document(id='5b208e74-df21-4596-940a-a033f830f77e', metadata={'source': 'wiki/faiss'}, page_content='FAISS 是向量检索库'),
  np.float32(0.7203966)),
 (Document(id='884c22ba-1443-4835-b363-11bbb41b6811', metadata={'source': 'wiki/lc'}, page_content='LangChain 是 LLM 编排框架'),
  np.float32(0.416677))]

In [16]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
docs = retriever.invoke("什么是向量检索")
docs

[Document(id='5b208e74-df21-4596-940a-a033f830f77e', metadata={'source': 'wiki/faiss'}, page_content='FAISS 是向量检索库'),
 Document(id='884c22ba-1443-4835-b363-11bbb41b6811', metadata={'source': 'wiki/lc'}, page_content='LangChain 是 LLM 编排框架')]